# Parse predicted labels

In [22]:
import numpy as np
import pandas as pd

### Load the questions and labels

Two question sets are in use:
- `MedChangeQA.csv`, the authors' released file. Its questions are paired with the wrong labels (see `compare_medchange_datasets.ipynb`), so scores on it are not meaningful.
- `medchange_subset.csv`, rebuilt from MedRevQA with the authors' grouping logic (`repro/build_subset.py`). It pairs each question with its own labels.

Each answer file is scored against the question set it actually answered.

In [23]:
df = pd.read_csv("../Datasets/MedChangeQA.csv", index_col=0)
subset = pd.read_csv("../repro/data/medchange_subset.csv").rename(
    columns={"question": "Question", "latest_label": "Newest Label", "outdated_label": "Outdated Label"})

COLUMNS = ["Question", "Newest Label", "Outdated Label"]
datasets = {
    "MedChangeQA.csv": df[COLUMNS].reset_index(drop=True),
    "subset": subset[COLUMNS],
}
{name: len(d) for name, d in datasets.items()}

{'MedChangeQA.csv': 512, 'subset': 512}

In [24]:
def parse_labels(lines):
    # copied unchanged from get_predictions() in the authors' experiments_results.ipynb
    llm_predicted_labels = list()
    for line in lines:
        if "SUPPORTED" in line:
            llm_predicted_labels.append("SUPPORTED")
        elif "REFUTED" in line:
            llm_predicted_labels.append("REFUTED")
        elif "NOT ENOUGH INFORMATION" in line:
            llm_predicted_labels.append("NOT ENOUGH INFORMATION")
    return llm_predicted_labels

### Parse each model's answers

In [25]:
# generated answers: line i = answer to row i of the question set given next to the file.

filenames = {
    "qwen25-7b": ("output/qwen25-7b_answers.txt", "MedChangeQA.csv"),
    "qwen25-7b bf16 (medchange subset)": ("output/qwen_bf16_answers.txt", "subset"),
    "biomistral-7b": ("output/biomistral-7b_answers.txt", "MedChangeQA.csv"),
    "biomistral-7b (new)": ("output/biomistral-7b_answers_(new)", "MedChangeQA.csv"),
    "biomistral-7b (medchange subset)": ("output/biomistral-7b_medchange_subset.txt", "subset"),
    
}

LABELS = ["SUPPORTED", "REFUTED", "NOT ENOUGH INFORMATION"]

parsed = {name: {} for name in datasets}
for model, (filename, dataset) in filenames.items():
    with open(filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    predictions = parse_labels(lines)

    unparsed = [i for i, line in enumerate(lines) if not any(l in line for l in LABELS)]
    multi = [i for i, line in enumerate(lines) if sum(l in line for l in LABELS) > 1]

    print(f"{model} [{dataset}]: {len(lines)} answers -> {len(predictions)} labels")
    print(f"  answers with no label (skipped by the rule): {unparsed}")
    print(f"  answers mentioning more than one label (first match in rule order wins): {multi}")
    print(f"  label counts: {pd.Series(predictions).value_counts().to_dict()}")

    if len(predictions) == len(datasets[dataset]):
        parsed[dataset][model] = predictions
    else:
        print(f"  !! number of labels does not match the {len(datasets[dataset])} questions -> not aligned, left out")
    print()

qwen25-7b [MedChangeQA.csv]: 512 answers -> 512 labels
  answers with no label (skipped by the rule): []
  answers mentioning more than one label (first match in rule order wins): [344]
  label counts: {'SUPPORTED': 379, 'NOT ENOUGH INFORMATION': 114, 'REFUTED': 19}

qwen25-7b bf16 (medchange subset) [subset]: 512 answers -> 512 labels
  answers with no label (skipped by the rule): []
  answers mentioning more than one label (first match in rule order wins): []
  label counts: {'SUPPORTED': 394, 'NOT ENOUGH INFORMATION': 72, 'REFUTED': 46}

biomistral-7b [MedChangeQA.csv]: 512 answers -> 503 labels
  answers with no label (skipped by the rule): [61, 103, 118, 328, 371, 387, 440, 494, 509]
  answers mentioning more than one label (first match in rule order wins): []
  label counts: {'NOT ENOUGH INFORMATION': 464, 'SUPPORTED': 38, 'REFUTED': 1}
  !! number of labels does not match the 512 questions -> not aligned, left out

biomistral-7b (new) [MedChangeQA.csv]: 512 answers -> 512 labels

### Save the parsed labels

One file per question set: `parsed_labels.csv` (MedChangeQA.csv questions and labels) and `parsed_labels_subset.csv` (subset questions and labels).

In [26]:
outfiles = {"MedChangeQA.csv": "output/parsed_labels.csv", "subset": "output/parsed_labels_subset.csv"}

for dataset, predictions_by_model in parsed.items():
    out = datasets[dataset].copy()
    for model, predictions in predictions_by_model.items():
        out[model] = predictions
    out.to_csv(outfiles[dataset], index=False)
    print(outfiles[dataset])
    display(out.head())

output/parsed_labels.csv


,Question,Newest Label,Outdated Label,qwen25-7b,biomistral-7b (new)
0,Is cranberry juice effective for treating urin...,SUPPORTED,REFUTED,SUPPORTED,SUPPORTED
1,Is add-on lamotrigine effective in reducing se...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
2,Do statins help reduce cardiovascular events i...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
3,Do antibiotics effectively reduce pain in chil...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,NOT ENOUGH INFORMATION
4,Does cell salvage reduce the need for blood tr...,REFUTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED


output/parsed_labels_subset.csv


,Question,Newest Label,Outdated Label,qwen25-7b bf16 (medchange subset),biomistral-7b (medchange subset)
0,Are point-of-care tests effective for diagnosi...,SUPPORTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
1,Are antibody tests effective in diagnosing cur...,SUPPORTED,REFUTED,SUPPORTED,SUPPORTED
2,Is convalescent plasma effective and safe for ...,REFUTED,NOT ENOUGH INFORMATION,SUPPORTED,SUPPORTED
3,Is there a beneficial effect of the interventi...,SUPPORTED,NOT ENOUGH INFORMATION,NOT ENOUGH INFORMATION,SUPPORTED
4,Is there a beneficial effect of the interventi...,SUPPORTED,NOT ENOUGH INFORMATION,NOT ENOUGH INFORMATION,SUPPORTED
